# Master Pipeline Orchestrator

> **Purpose**: Run the complete end-to-end data quality pipeline in one go by calling `main.run_pipeline()`.

**All orchestration logic lives in `../main.py`. This notebook only imports and calls `run_pipeline()`.**

### Pipeline stages
```
Load Dataset (CSV)
      ↓
Cleaning          — dates, prices, categoricals          (cleaning.py)
      ↓
Validation        — nulls, schema, dates, payments, regex (validation.py)
      ↓
Business Rules    — negatives, duplicates, categories    (rules.py)
      ↓
Statistics        — descriptive stats, Z-score, IQR      (statistics.py)
      ↓
Anomaly Detection — Isolation Forest + LOF consensus     (anomaly.py)
      ↓
Quality Scoring   — completeness × 0.40 + ...            (scoring.py)
      ↓
Reports           — Excel (6 sheets) + JSON              (reports.py)
```

> **CLI alternative**: `python main.py` from the `ml_engine/` directory

In [ ]:
import sys
import os
import json
import time

sys.path.insert(0, os.path.abspath(".."))

from main import run_pipeline

DATA_PATH   = os.path.join("..", "data", "dataset_ecommerce_transactions_data.csv")
REPORTS_DIR = os.path.join("..", "reports")

print(f"Data path    : {os.path.abspath(DATA_PATH)}")
print(f"Reports dir  : {os.path.abspath(REPORTS_DIR)}")

## Run the Complete Pipeline

In [ ]:
print("Starting pipeline...")
wall_start = time.perf_counter()

pipeline_result = run_pipeline(
    data_path=DATA_PATH,
    output_dir=REPORTS_DIR,
    generate_reports=True,
)

wall_elapsed = round(time.perf_counter() - wall_start, 2)
print(f"\nPipeline completed in {wall_elapsed}s — Status: {pipeline_result.get('status', 'unknown').upper()}")
print(f"Reported total time (from stages): {pipeline_result.get('total_pipeline_time_sec', 0.0):.2f}s")

## Pipeline Stage Summary

In [ ]:
import pandas as pd

summary_rows = []
for stage, info in pipeline_result.get("stages", {}).items():
    summary_rows.append({
        "Stage":    stage,
        "Status":   "OK" if info.get("error") is None else "FAILED",
        "Time (s)": info.get("elapsed_sec", 0),
        "Error":    info.get("error", "") or "",
    })

pd.DataFrame(summary_rows)

## Quality Scores

In [ ]:
scoring_info = pipeline_result.get("stages", {}).get("scoring", {})
score_fields = ["dataset_score", "completeness_score", "uniqueness_score",
                "validity_score", "rules_quality_score", "anomaly_penalty"]

print("=== Data Quality Scores ===")
for k in score_fields:
    v = scoring_info.get(k, "N/A")
    unit = "%" if "penalty" in k else ""
    print(f"  {k:<25}: {v}{unit}")

## Business Rule Violations

In [ ]:
rules_info = pipeline_result.get("stages", {}).get("business_rules", {})
print(f"Violation types    : {rules_info.get('violation_types', 0)}")
print(f"Affected records   : {rules_info.get('total_affected_records', 0):,}")
print()

for v in rules_info.get("violations", []):
    print(f"  [{v['severity']:6}] {v['rule']:<50} {v['count']:>7,} rows ({v['percentage']:.1f}%)")

## Validation Summary

In [ ]:
val_summary = pipeline_result.get("stages", {}).get("validation", {}).get("summary", {})
print("=== Validation Summary ===")
for k, v in val_summary.items():
    print(f"  {k:<20}: {v}")

## Generated Report Paths

In [ ]:
report_info = pipeline_result.get("stages", {}).get("reports", {})
exported    = report_info.get("result", {}).get("exported", {})

for fmt, path in exported.items():
    print(f"  [{fmt.upper()}] {path}")

---
## Key Takeaways

- `run_pipeline()` is the single entry point for the complete pipeline — same logic as `python main.py`
- The result dict has `total_pipeline_time_sec` so you can track performance over time
- All stages are isolated — a failure in one stage does not crash the next stage
- The `scoring` stage now reports `rules_quality_score` and `anomaly_penalty` in addition to the main scores
- Reports are timestamped so each run produces a new file without overwriting previous results
- To only run specific stages, import individual module functions (e.g. `from cleaning import run_cleaning`)